In [1]:
import pandas as pd
from pathlib import Path
import calendar

In [2]:
db_path = Path(r"D:\ProyectoAnalisisElectrico\MedidasValorizadas")
to_save_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")

In [3]:
def days_in_month(date):
    año = date//100
    mes = date%100
    _, dias = calendar.monthrange(2000+año, mes)
    return dias

In [4]:
group_data = [
    'clave',
    'nombre_barra',
    'tension',
    'Zona',
    'Razon_Social',
    'RUT',
    'Nombre_Corto',
    'Hora_Dia',
    'Año_Mes',
    'tipo'
    ]

In [7]:
for folder_date in db_path.iterdir():
    if not folder_date.is_dir():
        continue
        
    date = folder_date.name
    
    if int(date) < 2505: # Procesamos solo los datos con el formato nuevo en este codigo
        continue

    # Base de datos antigua
    to_save_folder = to_save_path / f"{date}"
    n_days = days_in_month(int(date))
    es_septiembre = date.endswith("09")
    
    if to_save_folder.is_dir():
        print(f"Los datos de {date} ya fueron procesados. Saltando...")
        continue  
        
    to_save_folder.mkdir(parents=True, exist_ok=True)

    file_medida = folder_date / f"{date}_medidas_horarias.parquet"

    print(f"Empezando {date} ")

    df = pd.read_parquet(file_medida)  
    
    df["Fecha_Medicion_last"] = pd.to_datetime(df["Fecha_Medicion_last"], format="%Y-%m-%d %H:%M:%S")
    df["Hora_Dia"] = df["Fecha_Medicion_last"].dt.hour
    df["Año_Mes"] = df["Fecha_Medicion_last"].dt.to_period('M').dt.to_timestamp()

    groups = df.groupby(by=group_data, sort=False) 
    groups_with_size = groups.size()
    good_groups_mask = good_groups_mask = (groups_with_size == n_days ) | ((groups_with_size == n_days  - 1) & es_septiembre) # Por el cambio de hora, generamos una hora sin consumo.
    
    agg_rules = {
    'medida_3_sum': lambda x: x.sum() / days_in_month(int(date)), 
    'CMg[CLP/KWh]_mean': "mean",
    'valorizado_CLP_sum': lambda x: x.sum() / days_in_month(int(date))
    }

    df_agregado = groups.agg(agg_rules)
    df_good = df_agregado[good_groups_mask].reset_index()
    df_good = df_good.rename(columns={
    'Hora_Dia': 'Hora',
    'medida_3_sum': 'medida',
    'CMg[CLP/KWh]_mean': 'CMg[CLP/KWh]',
    'valorizado_CLP_sum': 'valorizado_CLP'
    })
    df_good.to_parquet(to_save_folder / f"{date}_mean_month.parquet", engine="pyarrow", compression="snappy")

    bad_groups_mask = ~good_groups_mask
    if bad_groups_mask.any(): # Si al menos un grupo falló
        print(f"{folder_date.name} tuvo problemas con {file_medida}")
        df_bad = groups.filter(lambda x: len(x) != days_in_month(int(date)))
        df_bad.to_parquet(to_save_folder / f"{date}_errores_month.parquet", engine="pyarrow", compression="snappy")

    
      

Empezando 2505 
Empezando 2506 
Empezando 2507 
Empezando 2508 
Empezando 2509 
Empezando 2510 
Empezando 2511 
Empezando 2512 
Empezando 2601 
Empezando 2602 
Empezando 2603 
Empezando 2604 


Hubieron problemas con 2509 y con 2604 descubrí que es por los cambios de hora